# 06 - Spillover calibration and buffer choice

This notebook computes year-specific nearest-treated distances, assigns untreated cells to spillover rings, estimates first-pass HT/Hajek ring diagnostics, and records a conservative buffer choice for the direct ATT notebooks.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.estimation_workflow_utils import (
    append_tag_to_filename,
    compute_nearest_treated_exposure,
    ht_hajek_ring_diagnostics,
    infer_run_tag,
)

DATA_DIR = PROJECT_ROOT / 'data'
INTERMEDIATE_DIR = DATA_DIR / 'intermediate'
SPATIAL_DIR = INTERMEDIATE_DIR / 'spatial_structure'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'
for path in [INTERMEDIATE_DIR, TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)


## User configuration

The default is the 1km workflow. The panel currently starts in 2001; set `START_YEAR = 2000` only after adding the 2000 deforestation outcome.

In [ ]:
RUN_TAG = '1km'
START_YEAR = 2001
END_YEAR = None

TREATMENT_KEY = 'protected_area'
CELL_COL = 'cell_id'
YEAR_COL = 'year'
OUTCOME_COL = 'loss_m2'

PANEL_PATH = INTERMEDIATE_DIR / f'panel_treatment_{RUN_TAG}.parquet'
if not PANEL_PATH.exists():
    PANEL_PATH = INTERMEDIATE_DIR / 'panel_treatment.parquet'
CENTROID_PATH = SPATIAL_DIR / append_tag_to_filename('03_grid_centroids.parquet', RUN_TAG)
if not CENTROID_PATH.exists():
    CENTROID_PATH = SPATIAL_DIR / '03_grid_centroids.parquet'

RINGS_KM = [(0, 5), (5, 10), (10, 25), (25, 50)]
BUFFER_RADII_KM = [10, 25, 50]
REFERENCE_DISTANCE_KM = 50
CHOSEN_BUFFER_KM = 25

EXPOSURE_PANEL_NAME = '06_nearest_treated_exposure.parquet'
RING_DIAGNOSTIC_NAME = '06_ring_ht_hajek_diagnostics.csv'
BUFFER_DECISION_NAME = '06_buffer_choice_summary.json'
RING_FIG_NAME = '06_ring_hajek_diagnostics.png'

print('Panel:', PANEL_PATH)
print('Centroids:', CENTROID_PATH)


## Load panel and centroids

In [ ]:
panel_cols = [CELL_COL, YEAR_COL, OUTCOME_COL, f'treated_it_{TREATMENT_KEY}', 'treated_it', 'never_treated']
panel = pd.read_parquet(PANEL_PATH, columns=[c for c in panel_cols if c])
panel[CELL_COL] = panel[CELL_COL].astype('string')
panel[YEAR_COL] = pd.to_numeric(panel[YEAR_COL], errors='coerce').astype(int)
panel = panel[panel[YEAR_COL] >= START_YEAR].copy()
if END_YEAR is not None:
    panel = panel[panel[YEAR_COL] <= END_YEAR].copy()

treated_col = f'treated_it_{TREATMENT_KEY}' if f'treated_it_{TREATMENT_KEY}' in panel.columns else 'treated_it'
centroids = pd.read_parquet(CENTROID_PATH)
centroids[CELL_COL] = centroids[CELL_COL].astype('string')

years = sorted(panel[YEAR_COL].unique())
print('Rows:', f'{len(panel):,}')
print('Cells:', f'{panel[CELL_COL].nunique():,}')
print('Years:', min(years), '-', max(years))
print('Treatment column:', treated_col)


## Compute nearest-treated distance and ring exposure

This avoids materializing full 25km or 50km all-pairs graphs. For each year, it builds a KDTree on treated cells and queries the nearest treated cell for every grid cell.

In [ ]:
exposure = compute_nearest_treated_exposure(
    panel,
    centroids,
    cell_col=CELL_COL,
    year_col=YEAR_COL,
    treated_col=treated_col,
    rings_km=RINGS_KM,
    buffer_radii_km=BUFFER_RADII_KM,
    years=years,
)

exposure_path = INTERMEDIATE_DIR / append_tag_to_filename(EXPOSURE_PANEL_NAME, RUN_TAG)
exposure.to_parquet(exposure_path, index=False)
print('Saved:', exposure_path)
print(exposure.head().to_string(index=False))


## HT/Hajek ring diagnostics

These are calibration diagnostics, not the final causal spillover estimates. They compare untreated cells in each ring to untreated cells more than 50km from treated cells in the same year, using empirical exposure shares as simple inverse-probability weights.

In [ ]:
diagnostic_df = panel[[CELL_COL, YEAR_COL, OUTCOME_COL, treated_col]].merge(
    exposure[[CELL_COL, YEAR_COL, 'nearest_treated_distance_km', 'distance_ring']],
    on=[CELL_COL, YEAR_COL],
    how='left',
)
diagnostic_df = diagnostic_df[pd.to_numeric(diagnostic_df[treated_col], errors='coerce').fillna(0).astype(int) == 0].copy()
diagnostic_df['calibration_ring'] = diagnostic_df['distance_ring'].astype('string')
diagnostic_df.loc[diagnostic_df['nearest_treated_distance_km'] > REFERENCE_DISTANCE_KM, 'calibration_ring'] = f'gt_{REFERENCE_DISTANCE_KM}km'
reference_label = f'gt_{REFERENCE_DISTANCE_KM}km'

ring_diag = ht_hajek_ring_diagnostics(
    diagnostic_df.dropna(subset=['calibration_ring']),
    outcome_col=OUTCOME_COL,
    year_col=YEAR_COL,
    ring_col='calibration_ring',
    reference_label=reference_label,
)
ring_diag_path = TABLE_DIR / append_tag_to_filename(RING_DIAGNOSTIC_NAME, RUN_TAG)
ring_diag.to_csv(ring_diag_path, index=False)
print('Saved:', ring_diag_path)
print(ring_diag.head(12).to_string(index=False))


## Buffer decision memo

In [ ]:
fig_path = FIG_DIR / append_tag_to_filename(RING_FIG_NAME, RUN_TAG)
if not ring_diag.empty:
    plot_df = ring_diag.pivot_table(index='year', columns='ring', values='hajek_difference', aggfunc='mean')
    ax = plot_df.plot(figsize=(11, 5), linewidth=1.6)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('Hajek ring diagnostics relative to >50km untreated cells')
    ax.set_ylabel('Difference in annual forest loss (m2)')
    ax.set_xlabel('Year')
    ax.grid(axis='y', alpha=0.3)
    ax.xaxis.grid(False)
    ax.legend(frameon=False, fontsize=9)
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(fig_path, dpi=220, bbox_inches='tight')
    plt.show()

decision = {
    'run_tag': RUN_TAG,
    'treatment_key': TREATMENT_KEY,
    'start_year': START_YEAR,
    'end_year': END_YEAR,
    'rings_km': RINGS_KM,
    'candidate_buffers_km': BUFFER_RADII_KM,
    'chosen_buffer_km': CHOSEN_BUFFER_KM,
    'reference_distance_km': REFERENCE_DISTANCE_KM,
    'exposure_panel_path': str(exposure_path),
    'ring_diagnostic_path': str(ring_diag_path),
    'interpretation': 'Chosen buffer is a design choice to be validated in notebook 07; the default 25km buffer is conservative while avoiding a 50km-only donor pool unless diagnostics require it.',
}
decision_path = TABLE_DIR / append_tag_to_filename(BUFFER_DECISION_NAME, RUN_TAG)
with open(decision_path, 'w', encoding='utf-8') as f:
    json.dump(decision, f, indent=2)
print('Saved:', decision_path)
print(json.dumps(decision, indent=2))
